# AgentWonder — End-to-End Flow Test

This notebook exercises the full AgentWonder pipeline:

1. **Load** registries from `config/`
2. **Validate** a workflow YAML into Pydantic models
3. **Cross-validate** all references
4. **Resolve** tool, prompt, policy, and template references
5. **Build** a `RuntimePlan`
6. **Execute** the workflow via the async executor
7. **Inspect** run status, outputs, and trace events

## 1. Setup — Load Registries

In [1]:
import sys, os
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")

from agentwonder.registry import TemplateRegistry, ToolRegistry, PromptRegistry, PolicyRegistry

config_dir = Path("config")

template_reg = TemplateRegistry()
template_reg.load_from_directory(config_dir / "templates")

tool_reg = ToolRegistry()
tool_reg.load_from_directory(config_dir / "tools")

prompt_reg = PromptRegistry()
prompt_reg.load_from_directory(config_dir / "prompts")

policy_reg = PolicyRegistry()
policy_reg.load_from_directory(config_dir / "policies")

print(f"Templates: {[t.id for t in template_reg.list_all()]}")
print(f"Tools:     {[t.id for t in tool_reg.list_all()]}")
print(f"Prompts:   {[t.id for t in prompt_reg.list_all()]}")
print(f"Policies:  {[t.id for t in policy_reg.list_all()]}")

Working directory: /Users/krishnakumar/AgentWorld/AgentWonder
Templates: ['evaluator_loop', 'parallel_fanout_aggregate', 'router_specialists', 'sequential_with_approval', 'single_agent_with_tools']
Tools:     ['classify_ticket', 'fetch_bonds', 'fetch_break_details', 'fetch_equities', 'fetch_fx', 'fetch_trade_context', 'llm_classifier', 'llm_summarizer', 'log_interaction', 'lookup_customer', 'post_resolution_update', 'publish_content', 'suggest_resolution']
Prompts:   ['content_generation_prompt', 'propose_resolution_prompt']
Policies:  ['checker_signoff', 'content_publish_signoff']


## 2. Load and Validate Workflow YAML

In [2]:
from agentwonder.compiler.loader import load_yaml
from agentwonder.compiler.validators import validate_workflow, cross_validate_workflow

raw = load_yaml(config_dir / "workflows" / "break_resolution_v1.yaml")
print("Raw YAML keys:", list(raw.keys()))

workflow = validate_workflow(raw)
print(f"\nValidated: {workflow.id} v{workflow.version}")
print(f"Template:  {workflow.template}")
print(f"Steps:     {[s.id for s in workflow.steps]}")
print(f"Tool refs: {workflow.tool_refs}")

Raw YAML keys: ['id', 'name', 'version', 'owner_team', 'template', 'description', 'models', 'tool_refs', 'inputs', 'steps', 'output', 'evals']

Validated: break_resolution_v1 v1.0.0
Template:  sequential_with_approval
Steps:     ['enrich_break', 'enrich_trade_context', 'propose_resolution', 'checker_approval', 'update_downstream']
Tool refs: ['fetch_break_details', 'fetch_trade_context', 'suggest_resolution', 'post_resolution_update']


## 3. Cross-Validate References

In [3]:
tools_dict = {t.id: t for t in tool_reg.list_all()}
templates_dict = {t.id: t for t in template_reg.list_all()}
prompts_dict = {t.id: t for t in prompt_reg.list_all()}
policies_dict = {t.id: t for t in policy_reg.list_all()}

errors = cross_validate_workflow(workflow, tools_dict, templates_dict, policies_dict, prompts_dict)

if errors:
    print("Cross-validation FAILED:")
    for e in errors:
        print(f"  - {e}")
else:
    print("Cross-validation PASSED — all references valid")

Cross-validation PASSED — all references valid


## 4. Resolve References and Build RuntimePlan

In [4]:
from agentwonder.compiler.resolver import resolve_workflow
from agentwonder.compiler.builder import build_plan

resolved = resolve_workflow(workflow, tools_dict, templates_dict, prompts_dict, policies_dict)
print(f"Resolved workflow: {resolved.workflow.id}")
print(f"  Template: {resolved.template.id} v{resolved.template.version}")
print(f"  Tools:    {sorted(resolved.tools.keys())}")
print(f"  Prompts:  {sorted(resolved.prompts.keys())}")
print(f"  Policies: {sorted(resolved.policies.keys())}")

plan = build_plan(resolved)
print(f"\nRuntimePlan built:")
print(f"  Workflow:        {plan.workflow_id} v{plan.workflow_version}")
print(f"  Execution order: {plan.execution_order}")
print(f"  Parallel groups: {plan.parallel_groups}")
print(f"  Approval gates:  {plan.approval_step_ids}")
print(f"  Total steps:     {len(plan.steps)}")

Resolved workflow: break_resolution_v1
  Template: sequential_with_approval v1.0.0
  Tools:    ['fetch_break_details', 'fetch_trade_context', 'post_resolution_update', 'suggest_resolution']
  Prompts:  ['propose_resolution_prompt']
  Policies: ['checker_signoff']

RuntimePlan built:
  Workflow:        break_resolution_v1 v1.0.0
  Execution order: ['enrich_break', 'enrich_trade_context', 'propose_resolution', 'checker_approval', 'update_downstream']
  Parallel groups: [['enrich_break'], ['enrich_trade_context'], ['propose_resolution'], ['checker_approval'], ['update_downstream']]
  Approval gates:  ['checker_approval', 'update_downstream']
  Total steps:     5


## 5. Execute the Workflow

In [5]:
import asyncio
from agentwonder.runtime.executor import WorkflowExecutor
from agentwonder.schemas.run import RunRequest

executor = WorkflowExecutor()
request = RunRequest(
    workflow_id="break_resolution_v1",
    inputs={"break_id": "BRK-42", "source_system": "operations_hub"},
    environment="dev",
    requester="notebook_test",
)

status = await executor.execute(request, plan)

print(f"Run ID:    {status.run_id}")
print(f"State:     {status.state.value}")
print(f"Started:   {status.started_at}")
print(f"Completed: {status.completed_at}")
print(f"Error:     {status.error}")

GOOGLE_API_KEY not set — LLM calls will use stub responses


Run ID:    6eafa35b1dae
State:     completed
Started:   2026-04-05 12:50:57.462363+00:00
Completed: 2026-04-05 12:50:57.463007+00:00
Error:     None


## 6. Inspect Step Outputs

In [6]:
import json

print("Step outputs:\n")
for step_id, output in status.outputs.items():
    print(f"--- {step_id} ---")
    print(json.dumps(output, indent=2, default=str))
    print()

Step outputs:

--- enrich_break ---
{
  "tool_id": "fetch_break_details",
  "tool_type": "rest",
  "status": "success",
  "output": "Stub result from tool 'fetch_break_details'"
}

--- enrich_trade_context ---
{
  "tool_id": "fetch_trade_context",
  "tool_type": "rest",
  "status": "success",
  "output": "Stub result from tool 'fetch_trade_context'"
}

--- propose_resolution ---
{
  "model": "gemini-2.5-flash",
  "prompt_snippet": "You are an operations resolution assistant.\nUse the provided break context and trade context.\nPropos",
  "tools_available": [
    "suggest_resolution"
  ],
  "status": "success",
  "output": "[Stub LLM response for step 'propose_resolution']"
}

--- checker_approval ---
{
  "approval_id": "8cad4ccd2f18",
  "outcome": "approved",
  "decided_by": "system_auto_approve_v1"
}

--- update_downstream ---
{
  "tool_id": "post_resolution_update",
  "tool_type": "rest",
  "status": "success",
  "output": "Stub result from tool 'post_resolution_update'"
}



## 7. Inspect Trace Events

In [7]:
session = executor.session_store.get_session(status.run_id)
trace_events = session["data"].get("trace_events", [])

print(f"Total trace events: {len(trace_events)}\n")
for evt in trace_events:
    step = evt.get("step_id") or "(run-level)"
    print(f"  [{evt['event_type']:>20}]  step={step:<25}  data_keys={list(evt.get('data', {}).keys())}")

Total trace events: 18

  [           run_start]  step=(run-level)                data_keys=['workflow_id', 'workflow_version', 'template_id', 'model_default', 'model_evaluator', 'llm_configured']
  [          step_start]  step=enrich_break               data_keys=['step_type', 'model', 'prompt_version']
  [        tool_invoked]  step=enrich_break               data_keys=['tool_id', 'tool_version', 'tool_type', 'method', 'endpoint', 'inputs']
  [            step_end]  step=enrich_break               data_keys=['step_type', 'duration_ms', 'result_keys']
  [          step_start]  step=enrich_trade_context       data_keys=['step_type', 'model', 'prompt_version']
  [        tool_invoked]  step=enrich_trade_context       data_keys=['tool_id', 'tool_version', 'tool_type', 'method', 'endpoint', 'inputs']
  [            step_end]  step=enrich_trade_context       data_keys=['step_type', 'duration_ms', 'result_keys']
  [          step_start]  step=propose_resolution         data_keys=['step_type

## 8. Verify — Assertions

In [8]:
from agentwonder.schemas.common import RunState

assert status.state == RunState.COMPLETED, f"Expected COMPLETED, got {status.state}"
assert status.error is None, f"Unexpected error: {status.error}"
assert len(status.outputs) == 5, f"Expected 5 step outputs, got {len(status.outputs)}"
assert "enrich_break" in status.outputs
assert "checker_approval" in status.outputs
assert status.outputs["checker_approval"]["outcome"] == "approved"
assert len(trace_events) > 0

print("All assertions passed!")
print(f"  - State: {status.state.value}")
print(f"  - Steps completed: {len(status.outputs)}")
print(f"  - Trace events: {len(trace_events)}")
print(f"  - Approval outcome: {status.outputs['checker_approval']['outcome']}")

All assertions passed!
  - State: completed
  - Steps completed: 5
  - Trace events: 18
  - Approval outcome: approved
